**Installing Dependencies**

In [ ]:
!pip install -U dspy
!pip install datasets
!pip install torch transformers

**Setting Up Ollama**

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
process = subprocess.Popen("ollama serve", shell=True)

In [ ]:
!ollama pull llama3.1:8b

**Import Libraries**

In [ ]:
import json
import random
import re
import time
from datasets import load_dataset
import dspy
import numpy as np
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

**Configure DSPy & LLM**

In [ ]:
# === Configure DSPy with Ollama LLM ===
lm = dspy.LM('ollama_chat/llama3.1:8b', api_base='http://localhost:11434', api_key='')
dspy.configure(lm=lm)

**Load Training Datasets**

In [ ]:
# === Load GSM8K and HotpotQA for training/testing ===
print("Loading datasets...")
gsm8k = load_dataset("gsm8k", "main", split="train[:10]")
hotpotqa = load_dataset("hotpot_qa", "fullwiki", split="train[:10]", trust_remote_code=True)

gsm8k_list = list(gsm8k)
hotpotqa_list = list(hotpotqa)

# Combine prompts and ground truths
data = [(ex["question"], ex["answer"].split("####")[-1].strip()) for ex in gsm8k_list] \
     + [(ex["question"], ex["answer"]) for ex in hotpotqa_list]

random.shuffle(data)
prompts, ground_truths = zip(*data)

prompts = list(prompts)
ground_truths = list(ground_truths)

**Action Space & Pool**

In [ ]:
# === Define Action Space ===
MODULES = ["CoT", "Predict"]
SIGNATURES = [
    "question -> answer", "text -> summary", "question -> reasoning", "question -> hypothesis",
    "problem -> solution", "problem_description -> explanation", "context -> summary", "context -> briefing",
    "word_problem -> solution", "math_problem -> answer", "prompt -> response", "query -> response",
    "text -> response", "prompt -> generated_text", "query -> generated_text"
]
ACTIONS = MODULES + SIGNATURES + ["stop"]

**Policy Model Setup**

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

policy_model = model
policy_model.config.pad_token_id = tokenizer.pad_token_id
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_model.to(device)

print(f"Using device: {device}")
print("GPT-2 model loaded successfully")

**Helper Functions**

In [ ]:
# ============================================================
# Compare Answers Functions
def compare_answers(response, ground_truth):
    """
    Extract and compare the final answer from response and ground truth.
    """
    if response is None:
        return 0.0

    response_bracket_matches = re.findall(r'\[(.*?)\]', str(response))
    if response_bracket_matches:
        response_answer = response_bracket_matches[-1].strip()
    else:
        response_num_matches = re.findall(r'-?\d*\.?\d+', str(response))
        response_answer = response_num_matches[-1] if response_num_matches else None

    gt_bracket_matches = re.findall(r'\[(.*?)\]', str(ground_truth))
    if gt_bracket_matches:
        gt_answer = gt_bracket_matches[-1].strip()
    else:
        gt_num_matches = re.findall(r'-?\d*\.?\d+', str(ground_truth))
        gt_answer = gt_num_matches[-1] if gt_num_matches else None

    if response_answer and gt_answer:
        response_num_match = re.search(r'-?\d*\.?\d+', response_answer)
        gt_num_match = re.search(r'-?\d*\.?\d+', gt_answer)
        if response_num_match and gt_num_match:
            return 1.0 if response_num_match.group(0) == gt_num_match.group(0) else 0.0
        else:
            return 1.0 if response_answer == gt_answer else 0.0
    return 0.0

# ============================================================
# Compute Reward Function
def compute_reward(prompt, response, ground_truth):
    """
    Computes a reward for the response based on ground truth matching.
    Falls back to LLM judge evaluation if exact match score is 0.0.
    """
    if response is None:
        return 0.0

    score = compare_answers(response, ground_truth)
    if score == 0.0:
        eval_prompt = f"""Evaluate whether the following response correctly answers the prompt based on the ground truth and give your final score in the square brackets [final score]. Only the value in the [] should be in your response and nothing else.
Prompt: {prompt}
Response: {response}
Ground Truth: {ground_truth}
Return a score in range between 1.0 to 0.0 if the response is correct or partially correct (matches or is equivalent to the ground truth), or 0.0 if incorrect.
final score:[]"""
        try:
            llm_response = lm(eval_prompt)
            raw_text = llm_response[0] if isinstance(llm_response, list) and len(llm_response) > 0 else str(llm_response)
            bracket_matches = re.findall(r'\[(.*?)\]', raw_text)
            if bracket_matches:
                score_str = bracket_matches[-1].strip()
            else:
                num_matches = re.findall(r'-?\d*\.?\d+', raw_text)
                score_str = num_matches[-1] if num_matches else "0.0"
            score = float(score_str)
            return min(max(score, 0.0), 1.0)
        except (ValueError, TypeError, IndexError):
            return 0.0

    return score

# ============================================================
# Generate Pipeline Function
def generate_pipeline(prompt, model, max_steps=3):
    """
    Generate a discrete DSPy pipeline using action sampling from the policy model.
    """
    device = next(model.parameters()).device
    state = {"prompt": prompt, "partial_pipeline": []}
    actions_taken = []
    log_probs = []

    for _ in range(max_steps):
        input_text = f"Prompt: {state['prompt']} Pipeline: {' '.join(state['partial_pipeline'])}"
        inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True).to(device)
        outputs = model(**inputs)
        logits = outputs.logits[:, -1, :]

        if not state["partial_pipeline"]:
            valid_actions = MODULES
        elif len(state["partial_pipeline"]) == 1 and state["partial_pipeline"][0] in MODULES:
            valid_actions = SIGNATURES
        elif len(state["partial_pipeline"]) == 2:
            valid_actions = ["stop"]
        else:
            break

        valid_token_ids = []
        for a in valid_actions:
            tokens = tokenizer.encode(a, add_special_tokens=False)
            if len(tokens) == 0:
                continue
            valid_token_ids.append(tokens[0])

        action_logits = logits[0, valid_token_ids]
        action_probs = torch.softmax(action_logits, dim=-1)
        action_choice = torch.multinomial(action_probs, 1).item()
        action = valid_actions[action_choice]
        log_prob = torch.log(action_probs[action_choice])

        actions_taken.append(action)
        log_probs.append(log_prob)
        state["partial_pipeline"].append(action)

        if action == "stop":
            break

    return state["partial_pipeline"], actions_taken, log_probs

# ============================================================
# Execute Pipeline Function
def execute_pipeline(prompt, pipeline):
    """
    Execute the DSPy pipeline using the dspy modules.
    """
    if len(pipeline) != 2 or pipeline[0] not in MODULES or pipeline[1] not in SIGNATURES:
        return None

    module, signature = pipeline

    if signature.count("->") != 1:
        return None

    match = re.match(r"\s*([a-zA-Z_ ]+)\s*->\s*([a-zA-Z_ ]+)\s*", signature)
    if not match:
        return None

    inputfield = match.group(1).strip().replace(" ", "_").lower()
    outputfield = match.group(2).strip().replace(" ", "_").lower()

    try:
        if module == "CoT":
            program = dspy.ChainOfThought(signature)
        else:
            program = dspy.Predict(signature)
        response = program(**{inputfield: f"system instruction: Must give your final answer in square brackets without fail! e.g., [final answer] like this: [5] \n prompt: {prompt}"})
        return response.get(outputfield)
    except Exception:
        return None

# ============================================================
# Testing The Trained Model Function
def test_model(model, prompt, max_steps=3):
    """
    Evaluate the model on a prompt by generating and executing a DSPy pipeline.
    """
    model.eval()
    with torch.no_grad():
        pipeline, _, _ = generate_pipeline(prompt, model, max_steps=max_steps)
        response = execute_pipeline(prompt, pipeline[:-1] if pipeline and pipeline[-1] == "stop" else pipeline)
    return response

**Training Model**

In [ ]:
def train_grpo(policy_model, num_episodes=500, learning_rate=1e-4, K=4, save_path=None):
    """
    Train the policy model using Group Relative Policy Optimization (GRPO).
    For each episode, generate K pipelines per prompt, normalize advantages within the group,
    and update the policy model.
    """
    device = next(policy_model.parameters()).device
    optimizer = torch.optim.Adam(policy_model.parameters(), lr=learning_rate)
    rewards = []
    losses = []

    for episode in range(num_episodes):
        idx = np.random.randint(len(prompts))
        prompt = prompts[idx]
        ground_truth = ground_truths[idx]

        pipelines = []
        actions_list = []
        log_probs_list = []
        rewards_list = []

        for _ in range(K):
            pipeline, actions, log_probs = generate_pipeline(prompt, policy_model)
            response = execute_pipeline(prompt, pipeline[:-1] if pipeline and pipeline[-1] == "stop" else pipeline)
            reward = compute_reward(prompt, response, ground_truth)
            pipelines.append(pipeline)
            actions_list.append(actions)
            log_probs_list.append(log_probs)
            rewards_list.append(reward)

        group_mean = np.mean(rewards_list)
        group_std = np.std(rewards_list)
        avg_reward = float(group_mean)
        rewards.append(avg_reward)

        if group_std > 1e-8:
            advantages = [(r - group_mean) / (group_std + 1e-8) for r in rewards_list]
            loss = torch.tensor(0.0, device=device)
            for k in range(K):
                advantage = advantages[k]
                for log_prob in log_probs_list[k]:
                    loss = loss - log_prob * advantage

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            print(f"Episode {episode + 1}/{num_episodes}, Prompt: {prompt[:20]}..., Avg Reward: {avg_reward:.3f}, Loss: {loss.item():.3f}")
        else:
            losses.append(0.0)
            print(f"Episode {episode + 1}/{num_episodes}, Prompt: {prompt[:20]}..., Avg Reward: {avg_reward:.3f}, Loss: 0.000")

    if save_path:
        torch.save(policy_model.state_dict(), save_path)
        print(f"Model saved to {save_path}")
    return policy_model

**Run Training**

In [ ]:
trained_model = train_grpo(policy_model, num_episodes=2, learning_rate=2e-5, K=4)

In [ ]:
def evaluate_module(dataset, model, query=False, output_file="evaluation_results.json"):
    """
    Evaluates the given model on a dataset of examples and optionally stores the results in a JSON file.
    """
    total_correct = 0
    results = []
    start_time = time.time()

    for i, example in enumerate(dataset):
        prompt = f"{example.context} \n\n {example.question} " if (query and hasattr(example, 'context') and example.context) else example.question
        prediction_text = test_model(model, prompt)

        if example.answer:
            is_correct = (
                prediction_text and prediction_text.strip().lower() == example.answer.strip().lower()
            ) or (
                prediction_text and example.answer.strip().lower() in prediction_text.strip().lower()
            )
        else:
            is_correct = False

        score = 1 if is_correct else 0

        if not is_correct:
            eval_prompt = f"""Evaluate whether the following response correctly answers the prompt based on the ground truth.
Return a score of 1.0 or 0.0 if the response is correct and 0 if incorrect.
Only return the score inside the square brackets [].

Prompt: {example.question}
Response: {prediction_text}
Ground Truth: {example.answer}
Final score:[]"""
            try:
                llm_res = lm(eval_prompt)
                raw_text = llm_res[0] if isinstance(llm_res, list) and len(llm_res) > 0 else str(llm_res)
                score = 1 if "1" in raw_text else 0
            except Exception:
                score = 0

        total_correct += score
        print(f"Example {i + 1}/{len(dataset)} | Score: {score} | Total Correct: {total_correct}")

        if output_file:
            result = {
                "question": example.question,
                "response": prediction_text,
                "ground_truth": example.answer if example.answer else "N/A",
                "score": score,
            }
            if query and hasattr(example, 'context'):
                result["context"] = example.context
            results.append(result)

    total_time = time.time() - start_time

    if output_file and results:
        with open(output_file, "w") as f:
            json.dump(results, f, indent=4)

    return total_time, total_correct

**Loading Test Datasets**

In [ ]:
gsm8k_test = load_dataset("gsm8k", "main", split="test[:10]")
hotpotqa_test = load_dataset("hotpot_qa", "fullwiki", split="train[20:30]", trust_remote_code=True)

gsm8k_test_data = [dspy.Example(question=ex['question'], answer=ex["answer"].split("####")[-1].strip(), task_type='math').with_inputs('question') for ex in gsm8k_test]
hotpotqa_test_data = [dspy.Example(question=ex['question'], answer=ex['answer'], context=ex["context"], task_type='qa').with_inputs('question') for ex in hotpotqa_test]

In [ ]:
# Testing HotpotQA
htotal_time, htotal_correct = evaluate_module(hotpotqa_test_data, policy_model, query=True)
print(f"HotPotQA - Total time: {htotal_time}, Total correct: {htotal_correct}")

In [ ]:
# Testing GSM8K
gtotal_time, gtotal_correct = evaluate_module(gsm8k_test_data, policy_model)
print(f"GSM8K - Total time: {gtotal_time}, Total correct: {gtotal_correct}")